### **Introdução ao Home Lab de Análise de Logs**

Neste laboratório vamos desenvolver um analisador simples de logs utilizando Python. O objetivo é simular tarefas comuns realizadas por analistas de segurança e profissionais de SOC.

Vamos trabalhar com leitura de arquivos de log, identificação de eventos suspeitos e geração de relatórios com base nas informações encontradas.

### **Conceitos utilizados neste laboratório**

Durante o desenvolvimento deste projeto serão aplicados os seguintes conceitos da linguagem Python:

Leitura de arquivos utilizando with open

Estruturas de repetição (for)

Estruturas condicionais (if)

Manipulação de strings

Expressões regulares (Regex)

Uso de bibliotecas

Contagem de eventos com dicionários

Estrutura de dados Counter

Geração de relatórios em CSV

Exportação de dados em JSON

In [18]:
# Conecta o Google Colab ao Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Caminho base da pasta onde estão os arquivos do projeto
BASE = "/content/drive/MyDrive/Python"

# Caminho do arquivo de log que vamos analisar
LOG_PATH = f"{BASE}/Sample.log"

# Pasta onde vamos salvar os resultados (CSV/JSON)
OUT_DIR = f"{BASE}/output"

import os

# Cria a pasta de saída caso ela ainda não exista
os.makedirs(OUT_DIR, exist_ok=True)

# Mostra os caminhos usados no script
print("LOG:", LOG_PATH)
print("OUT:", OUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LOG: /content/drive/MyDrive/Python/Sample.log
OUT: /content/drive/MyDrive/Python/output


In [24]:
# Bibliotecas usadas no parser
import re          # regex para detectar padrões no log
import json        # exportar resultados em JSON
import csv         # exportar resultados em CSV
from collections import Counter  # contar ocorrências automaticamente
from datetime import datetime    # trabalhar com timestamps


# Padrões de eventos que queremos detectar no log
PATTERNS = {

    # Falha de login SSH
    "ssh_failed": re.compile(
        r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}).*sshd Failed password.* from (?P<ip>\d+\.\d+\.\d+\.\d+)"
    ),

    # Login SSH bem-sucedido
    "ssh_accepted": re.compile(
        r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}).*sshd Accepted password.* for (?P<user>\w+) from (?P<ip>\d+\.\d+\.\d+\.\d+)"
    ),

    # Erro 404 do servidor web (possível scan)
    "nginx_404": re.compile(
    r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}).*nginx 404 GET (?P<path>\S+) from (?P<ip>\d+\.\d+\.\d+\.\d+)"
    ),

    # Comando executado com sudo
    "sudo_cmd": re.compile(
        r"(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}).*sudo user=(?P<user>\w+)\s+command=(?P<cmd>.+)$"
    ),

}

In [25]:
# Lista onde vamos armazenar todos os eventos detectados no log
events = []

# Counter é uma estrutura da biblioteca collections que conta ocorrências automaticamente
# Cada vez que um IP aparecer, o contador aumenta

ip_failed = Counter()
ip_accepted = Counter()
ip_404 = Counter()
paths_404 = Counter()
sudo_cmds = Counter()

# Abrimos o arquivo de log para leitura
# LOG_PATH contém o caminho do arquivo sample.log
# encoding evita erros de caracteres
# errors="ignore" ignora caracteres quebrados

with open(LOG_PATH, "r", encoding="utf-8", errors="ignore") as f:
  # Percorre o arquivo linha por linha
    for line in f:

      # remove espaços e quebras de linha no início/fim
        line = line.strip()

        # se a linha estiver vazia, pula para a próxima
        if not line:
            continue

# variável usada para saber se algum padrão foi detectado
        matched = False

# tenta encontrar o padrão ssh_failed na linha usando regex
        m = PATTERNS["ssh_failed"].search(line)
        if m:
            matched = True
            # extrai o IP capturado pela regex
            ip = m.group("ip")
            # aumenta contador de falhas para esse IP
            ip_failed[ip] += 1
            # registra no events[]
            events.append({"type":"ssh_failed", "ts":m.group("ts"), "ip":ip, "raw":line})

# tenta encontrar logins ssh aceitos usando regex
        m = PATTERNS["ssh_accepted"].search(line)

        if m:
            matched = True
            # pega o IP e o usuário da regex
            ip = m.group("ip")
            user = m.group("user")
            #conta logins aceitos por IP
            ip_accepted[ip] += 1
            #registra no events[]
            events.append({"type":"ssh_accepted", "ts":m.group("ts"), "ip":ip, "user":user, "raw":line})

# tenta encontrar erros 404 (scan web) usando regex
        m = PATTERNS["nginx_404"].search(line)
        if m:
            matched = True
            #pega o IP e o caminho e o endpoint acessado
            ip = m.group("ip")
            path = m.group("path")
            #conta quantos erros esse IP gerou
            ip_404[ip] += 1
            #conta qual endpoint foi mais acessado
            paths_404[path] += 1
            #registra no events[]
            events.append({"type":"nginx_404", "ts":m.group("ts"), "ip":ip, "path":path, "raw":line})

# tenta encontrando comandos sudo
        m = PATTERNS["sudo_cmd"].search(line)
        if m:
            matched = True
            #usuário que executou e o comando executado
            user = m.group("user")
            cmd = m.group("cmd").strip()
            #conta quantos comandos esse usuário executou
            sudo_cmds[cmd] += 1
            #registra no events[]
            events.append({"type":"sudo_cmd", "ts":m.group("ts"), "user":user, "cmd":cmd, "raw":line})

# caso nenhum padrão seja identificado
        if not matched:
          #salva no events[] como tipo unclassified
            events.append({"type":"unclassified", "raw":line})

len(events), events[:3]

(45,
 [{'type': 'ssh_failed',
   'ts': '2026-03-05 20:58:01',
   'ip': '191.10.20.30',
   'raw': '\ufeff2026-03-05 20:58:01 INFO sshd Failed password for invalid user admin from 191.10.20.30 port 50112'},
  {'type': 'ssh_failed',
   'ts': '2026-03-05 20:58:02',
   'ip': '191.10.20.30',
   'raw': '2026-03-05 20:58:02 INFO sshd Failed password for root from 191.10.20.30 port 50115'},
  {'type': 'ssh_failed',
   'ts': '2026-03-05 20:58:04',
   'ip': '191.10.20.30',
   'raw': '2026-03-05 20:58:04 INFO sshd Failed password for root from 191.10.20.30 port 50118'}])

In [26]:
# número mínimo de falhas SSH para considerar suspeito
FAILED_THRESHOLD = 5

# lista onde vamos armazenar os IPs suspeitos
suspicious = []

# percorre todos os IPs que tiveram falha de login
for ip, cnt in ip_failed.items():

    # se o número de falhas for maior ou igual ao limite
    if cnt >= FAILED_THRESHOLD:

        # adiciona esse IP na lista de suspeitos
        suspicious.append({
            "ip": ip,                          # endereço IP
            "failed_logins": cnt,               # número de falhas
            "had_success_login": (ip_accepted[ip] > 0),  # se conseguiu logar depois
            "risk": "HIGH" if ip_accepted[ip] > 0 else "MEDIUM"  # risco maior se houve login bem-sucedido
        })


# ordena os suspeitos por risco e número de falhas
suspicious.sort(key=lambda x: (x["risk"] != "HIGH", -x["failed_logins"]))


# resumo geral da análise
summary = {
    "total_events": len(events),                # número total de eventos analisados
    "top_failed_ips": ip_failed.most_common(10), # IPs com mais falhas SSH
    "top_accepted_ips": ip_accepted.most_common(10), # IPs com mais logins aceitos
    "top_404_ips": ip_404.most_common(10),      # IPs que mais geraram erro 404
    "top_404_paths": paths_404.most_common(10), # endpoints mais acessados com erro 404
    "top_sudo_commands": sudo_cmds.most_common(10), # comandos sudo mais executados
    "suspicious_ips": suspicious[:10],          # top 10 IPs suspeitos
}

# mostra o resumo final
summary

{'total_events': 45,
 'top_failed_ips': [('191.10.20.30', 6),
  ('203.44.10.55', 6),
  ('185.44.22.19', 4),
  ('88.22.11.90', 4)],
 'top_accepted_ips': [('191.10.20.30', 1), ('177.55.201.18', 1)],
 'top_404_ips': [('45.67.89.10', 8), ('91.200.13.77', 4)],
 'top_404_paths': [('/wp-login.php', 2),
  ('/phpmyadmin', 1),
  ('/.env', 1),
  ('/admin', 1),
  ('/backup.zip', 1),
  ('/wp-admin', 1),
  ('/xmlrpc.php', 1),
  ('/config.php', 1),
  ('/admin/login', 1),
  ('/db.sql', 1)],
 'top_sudo_commands': [('/usr/bin/apt update', 1),
  ('/usr/bin/apt upgrade', 1),
  ('/bin/cat /etc/shadow', 1),
  ('/usr/bin/nano /etc/passwd', 1),
  ('/usr/bin/systemctl restart nginx', 1)],
 'suspicious_ips': [{'ip': '191.10.20.30',
   'failed_logins': 6,
   'had_success_login': True,
   'risk': 'HIGH'},
  {'ip': '203.44.10.55',
   'failed_logins': 6,
   'had_success_login': False,
   'risk': 'MEDIUM'}]}

In [27]:
# caminhos dos arquivos de saída
OUT_JSON = f"{OUT_DIR}/summary.json"
OUT_CSV = f"{OUT_DIR}/events.csv"

# salva o resumo da análise em JSON
with open(OUT_JSON, "w", encoding="utf-8") as jf:
    json.dump(summary, jf, indent=2, ensure_ascii=False)

# cria a lista de colunas do CSV com base nas chaves dos eventos
fieldnames = sorted({k for e in events for k in e.keys()})

# escreve todos os eventos detectados em um arquivo CSV
with open(OUT_CSV, "w", newline="", encoding="utf-8") as cf:
    writer = csv.DictWriter(cf, fieldnames=fieldnames)
    writer.writeheader()  # escreve o cabeçalho
    for e in events:
        writer.writerow(e)  # escreve cada evento como uma linha

# mostra onde os arquivos foram gerados
print("Gerado:")
print("-", OUT_JSON)
print("-", OUT_CSV)

Gerado:
- /content/drive/MyDrive/Python/output/summary.json
- /content/drive/MyDrive/Python/output/events.csv


In [28]:
# função para mostrar os itens mais frequentes de um Counter
def show_top(counter, title, n=5):
    print("\n" + title)
    for item, count in counter.most_common(n):  # pega os n mais comuns
        print(f"- {item}: {count}")

# menu interativo
while True:

    print("\nMENU")
    print("1) Top IPs com falha SSH")
    print("2) Top IPs com 404 (scan web)")
    print("3) Mostrar IPs suspeitos")
    print("0) Sair")

    # lê a escolha do usuário
    choice = input("Escolha: ").strip()

    # opção 1: mostrar IPs com mais falhas SSH
    if choice == "1":
        show_top(ip_failed, "Top SSH Failed", 10)

    # opção 2: mostrar IPs que mais geraram erro 404
    elif choice == "2":
        show_top(ip_404, "Top Nginx 404", 10)

    # opção 3: listar IPs classificados como suspeitos
    elif choice == "3":
        print("\nSuspeitos (threshold >= 5):")
        for s in suspicious:
            print(f"- {s['ip']} | failed={s['failed_logins']} | success={s['had_success_login']} | risk={s['risk']}")

    # opção 0: encerra o programa
    elif choice == "0":
       break

    # qualquer outra entrada
    else:
        print("Opção inválida.")


MENU
1) Top IPs com falha SSH
2) Top IPs com 404 (scan web)
3) Mostrar IPs suspeitos
0) Sair
Escolha: 1

Top SSH Failed
- 191.10.20.30: 6
- 203.44.10.55: 6
- 185.44.22.19: 4
- 88.22.11.90: 4

MENU
1) Top IPs com falha SSH
2) Top IPs com 404 (scan web)
3) Mostrar IPs suspeitos
0) Sair
Escolha: 2

Top Nginx 404
- 45.67.89.10: 8
- 91.200.13.77: 4

MENU
1) Top IPs com falha SSH
2) Top IPs com 404 (scan web)
3) Mostrar IPs suspeitos
0) Sair
Escolha: 3

Suspeitos (threshold >= 5):
- 191.10.20.30 | failed=6 | success=True | risk=HIGH
- 203.44.10.55 | failed=6 | success=False | risk=MEDIUM

MENU
1) Top IPs com falha SSH
2) Top IPs com 404 (scan web)
3) Mostrar IPs suspeitos
0) Sair
Escolha: 0
